In [1]:
import numpy as np
import pandas as pd 
import json
from shapely.geometry import Point
import geopandas 

# Read data

In [2]:
path = 'data/raw_data/'
file_name = 'evolution_station.csv'

In [3]:
stations_evolution = pd.read_csv(path+file_name)

In [4]:
stations_evolution['start_date'] = pd.to_datetime(stations_evolution['start_date'])
stations_evolution['end_date'] = pd.to_datetime(stations_evolution['end_date'])

# Transform to json

In [5]:
stations = {}

for i in stations_evolution.index: 
    station_ref = stations_evolution.at[i, 'Nom de référence']
    if station_ref in stations.keys(): 
        if stations_evolution.at[i, 'start_date'] > stations[station_ref]['Ouverture']: 
            stations[station_ref]['Ouverture'] = stations_evolution.at[i, 'start_date']
            stations[station_ref]['Nom'] = stations_evolution.loc[i, 'Nom']
            stations[station_ref]['geometry'] = Point([stations_evolution.at[i, 'longitude'], 
                                                       stations_evolution.at[i, 'latitude']]
                                                     )
            stations[station_ref]['Fermeture'] = stations_evolution.at[i, 'end_date']
    else: 
        stations[station_ref] = {'Nom': stations_evolution.at[i, 'Nom'],
                                 'Ouverture': stations_evolution.at[i, 'start_date'],
                                 'geometry': Point([stations_evolution.at[i, 'longitude'], 
                                                    stations_evolution.loc[i, 'latitude']]
                                                  ),
                                 'Fermeture': stations_evolution.at[i, 'end_date']
                                }

In [6]:
df = geopandas.GeoDataFrame(stations).T
df['Fermeture'] = df['Fermeture'].astype(str)
df['Ouverture'] = df['Ouverture'].astype(str)

In [7]:
df.to_file("data/stations.geojson", driver='GeoJSON')